## Installation
Gymnasium isn't part of the Python standard library, so it needs to be installed once per environment (computer / virtual environment / notebook server). If you've already installed it, you can skip this cell — running `pip install` again is harmless, it'll just confirm it's already there.


In [ ]:
%pip install "gymnasium[classic-control]"

## Imports
Same idea as any Python script: we import the libraries we need before using them.

- `gymnasium` — the RL environment toolkit itself
- `numpy` — for arrays and numerical operations (Gymnasium observations come back as NumPy arrays)
- `matplotlib.pyplot` — for plotting our agent's learning progress, and for building our video clips


In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)

## Creating the Environment
We create environments with `gym.make(environment_id)`. The id is just a string Gymnasium looks up in its registry:`"Acrobot-v1"` here. (The `-v1` is a version number; environments occasionally get updated, and the version number tells you exactly which rules apply.)


In [ ]:
env = gym.make("Acrobot-v1")
env

In [ ]:
observation, info = env.reset(seed=SEED)

print("Observation:", observation)
print("Info:", info)


## Understanding the Observation
The printed array has **6 numbers**. Acrobot's documentation tells us what they mean:

| Index | Meaning |
|---|---|
| 0 | cos(theta1) — cosine of the first joint's angle |
| 1 | sin(theta1) — sine of the first joint's angle |
| 2 | cos(theta2) — cosine of the second joint's angle |
| 3 | sin(theta2) — sine of the second joint's angle |
| 4 | angular velocity of joint 1 |
| 5 | angular velocity of joint 2 |

Why sine and cosine instead of just the angle in radians? Because angles "wrap around" (359° is right next to 0°), which can confuse a learning algorithm. Sine and cosine together describe an angle without any wraparound discontinuity — this is a common trick in robotics and RL.

Notice that the observation is **6 continuous numbers** — not a single tidy integer. There are infinitely many possible combinations of 6 real numbers, which matters a lot once we get to building a Q-table in Section 7.

For now, let's confirm this programmatically using the environment's `observation_space`:


In [ ]:
print("Observation space:", env.observation_space)
print("Shape:", env.observation_space.shape)
print("Lower bounds:", env.observation_space.low)
print("Upper bounds:", env.observation_space.high)

## Understanding the Action Space
Acrobot has a small, **discrete** action space — only 3 possible actions:

| Action | Meaning |
|---|---|
| 0 | apply -1 torque (push one way) |
| 1 | apply 0 torque (do nothing) |
| 2 | apply +1 torque (push the other way) |


In [ ]:
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)

# .sample() picks a uniformly random action
print("A random action:", env.action_space.sample())

In [ ]:
observation, info = env.reset(seed=SEED)

action = env.action_space.sample()  # pick a random action
observation, reward, terminated, truncated, info = env.step(action)

print("Action taken:", action)
print("New observation:", observation)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)


## Running a Full Episode (Random Actions)
Let's put the loop together and run one entire episode, taking random actions the whole time, just to see how the environment behaves before any learning happens.

### Python refresher: `while` loop with a `break`
We don't know in advance exactly how many steps an episode will take, so a `for` loop with a fixed range isn't quite right here. Instead we use a `while True:` loop — "keep going forever" — combined with `break` to escape the loop the moment we're `done`. This pattern is common any time you're repeating something "until a condition becomes true," rather than a fixed number of times.


In [ ]:
observation, info = env.reset(seed=SEED)
total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()  # random action
    observation, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps += 1

    done = terminated or truncated
    if done:
        break

print(f"Episode finished after {steps} steps.")
print(f"Total reward: {total_reward}")
print(f"Succeeded (terminated early)? {terminated}")

Since Acrobot gives `-1` every step, the total reward is just the negative of the number of steps taken. A random policy almost never manages to swing the arm up high enough — so you'll typically see `steps = 500` (the truncation limit) and a total reward around `-500`. That's our baseline to beat.

Numbers like this are useful, but seeing the arm flail around is a lot more intuitive. Let's make a quick video clip of it.


## 9. Watching the Random Agent
To actually *see* the environment, we need to render it as images. There are a few ways Gymnasium can show us what's happening, set by the `render_mode` argument when we create the environment:

- `render_mode="human"` — pops open a live window (great locally, doesn't work in most notebook setups)
- `render_mode="rgb_array"` — hands back each frame as a plain NumPy array of pixel colors, which **we** can display however we like

We'll use `"rgb_array"` and collect the frames into a Python list as the episode runs. Each frame is just an image (a grid of numbers), so a list of frames is really a list of images — a short flipbook.

### Python refresher: building a list as you go
```python
frames = []          # start empty
frames.append(x)     # add one item at a time, inside a loop
```
This is one of the most common patterns in Python: start with an empty list, then `.append()` to it each time through a loop.

### Animating a flipbook with matplotlib
Once we have a list of frames, `matplotlib.animation.FuncAnimation` can step through them and `HTML(anim.to_jshtml())` renders the result as an interactive video player, right inside the notebook — play, pause, and scrub through frames, no external files involved.


In [ ]:
from IPython.display import HTML
from matplotlib import animation

def collect_random_episode_frames(seed=SEED, max_steps=500):
    """Run one epsiode with random actions and return the list of rendered frames."""
    render_env = gym.make("Acrobot-v1", render_mode="rgb_array")
    obs, info = render_env.reset(seed=seed)

    frames = [render_env.render()]
    for _ in range(max_steps):
        action = render_env.action_space.sample()  # random actions
        _, _, terminated, truncated, _ = render_env.step(action)
        frames.append(render_env.render())
        if terminated or truncated:
            break

    render_env.close()
    return frames

def make_animation(frames, title=""):
    """Turn a list of image frames into an inline, playable animation"""
    fig, ax = plt.subplots()
    ax.axis("off")
    if title:
        ax.set_title(title)
    img = ax.imshow(frames[0])

    def update(i):
        img.set_data(frames[i])
        return [img]

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=40, blit=True
    )
    plt.close(fig)
    return anim

random_frames = collect_random_episode_frames()
print(f"Collected {len(random_frames)} frames from the random-action episode.")

random_anim = make_animation(random_frames, title="Random actions")
HTML(random_anim.to_jshtml())

## From Continuous to Discrete: Building a Q-Table for Acrobot
A Q-table needs a finite set of rows — one per possible state. That's no problem when a state is already a single integer, but Acrobot's state is **6 continuous numbers**, and there are infinitely many possible combinations of those.

The fix is to **discretize** the state: chop each of the 6 dimensions into a fixed number of "bins," then describe any observation as *which bin it falls into* on each dimension. This turns the infinite continuous space into a finite grid of discrete states that a Q-table can actually have rows for.

### Python refresher: `np.linspace` and `np.digitize`
- `np.linspace(low, high, n)` creates `n` evenly spaced numbers between `low` and `high` — we'll use this to define the edges of our bins.
- `np.digitize(value, bin_edges)` tells you which bin a value falls into, given a list of bin edges.

```python
edges = np.linspace(-1, 1, 5)        # 4 bins between -1 and 1
np.digitize(0.3, edges)              # tells you which of the 4 bins 0.3 falls into
```

We'll discretize each of the 6 observation dimensions into a small number of bins. More bins = more precision but a much bigger (slower to learn) table; fewer bins = faster learning but coarser detail. This trade-off is a recurring theme in RL.


In [ ]:
# Number of bins per dimension
N_BINS = 6

obs_low = env.observation_space.low
obs_high = env.observation_space.high

print("Low:", obs_low)
print("High", obs_high)

bin_edges = [
    np.linspace(obs_low[i], obs_high[i], N_BINS - 1)
    for i in range(len(obs_low))
]

def discretize(observation):
    """Convery a continuous 6-value observation into a tulple of 6 bin indicies."""
    return tuple(
        int(np.digitize(observation[i], bin_edges[i]))
        for i in range(len(observation))
    )

obs, info = env.reset(seed=SEED)
print("Raw observation:  ", obs)
print("Discretized state:", discretize(obs))

In [ ]:
Q = {}

def get_q_values(state):
    """Return the Q-values for a state, creating a fresh all-zero row if we haven't 
    seen it before."""
    if state not in Q:
        Q[state] = np.zeroes(env.action_space.n)
    return Q[state]

In [ ]:
def epsilon_greedy_action(state, epsilon):
    if np.random.rand() < epsilon:
        return  env.action_space.sample()
    else:
        return int(np.argmax(get_q_values(state)))

## Hyperparameters
Four knobs control how training behaves:
- **alpha** — learning rate: how much each new experience updates our existing estimate
- **gamma** — discount factor: how much we value future rewards vs. immediate ones
- **epsilon** — exploration rate, starting high and decaying over time
- **episodes** — how many practice rounds to run


In [ ]:
alpha = 0.1  # learning rate
gamma = 0.99 # discount factor

epsilon = 1.0 # start by exploring 100% of the time
epsilon_min = 0.05 # never fully stop exploring
epsilon_decay = 0.9995 # multiply epsilon by this after every episode

episodes = 5000
max_steps_per_episode = 500 

episode_rewards = []

In [ ]:
for ep in range(episodes):
    obs, info = env.reset()
    state = discretize(obs)
    total_reward = 0

    for t in range(max_steps_per_episode):
        action = epsilon_greedy_action(state, epsilon)

        next_obs, reward, terminated, truncated, info = env.step(action)
        next_state = discretize(next_obs)
        done = terminated or truncated

        # Q-learning update
        best_next_value = np.max(get_q_values(next_state))
        td_target = reward + gamma * best_next_value
        td_error = td_target - get_q_values(state)[action]
        Q[state][action] += alpha * td_error

        state = next_state
        total_reward += reward

        if done:
            break

    episode_rewards.append(total_reward)

    # Decay epsilon, but never below epsilon_min
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (ep + 1) % 500 == 0:
        recent_avg = np.mean(episode_rewards[-500:])
        print(f"Episode {ep + 1}/{episodes} | epsilon={epsilon:.3f} | avg reward (last 500): {recent_avg:.1f}")